# EDU Bronze to Silver transformation

Conforms one Bronze source instance into the 24-table Silver model. The notebook uses deterministic keys, validates declared PK/FK and nullability rules before publication, and skips targets whose relevant Bronze Delta versions and transformation contract are unchanged.

In [ ]:
# Deployment parameters
bronze_lakehouse_name = "EDU_Bronze"
source_instance_id = "synthetic-district-v2"
force_reload = False
exit_notebook_on_success = True

## Dependencies and incremental checkpoints

In [ ]:
import hashlib
import json
import uuid
from datetime import datetime, timezone

from pyspark.sql import Column, DataFrame, Window, functions as F, types as T

run_id = str(uuid.uuid4())
transform_version = "1.6.0"


def mark_stage(stage: str) -> None:
    spark.createDataFrame(
        [(run_id, stage, datetime.now(timezone.utc).replace(tzinfo=None))],
        "run_id string, stage string, recorded_at_utc timestamp",
    ).write.format("delta").mode("append").saveAsTable("transform_diagnostic")


mark_stage("initialized")

In [ ]:
# Bronze tables required by each Silver target.
TARGET_SOURCES = {
    "District": ["powerschool_districts"],
    "School": ["powerschool_districts", "powerschool_schools"],
    "Student": ["powerschool_students"],
    "Teacher": ["powerschool_teachers", "powerschool_schools"],
    "Course": ["powerschool_courses"],
    "SchoolAddress": ["powerschool_districts", "powerschool_schools"],
    "AcademicTerm": ["powerschool_terms"],
    "Section": ["powerschool_sections", "powerschool_schools", "powerschool_courses", "powerschool_terms", "powerschool_enrollments"],
    "SchoolEnrollment": ["powerschool_enrollments", "powerschool_reenrollments", "powerschool_students", "powerschool_schools"],
    "SectionEnrollment": ["powerschool_section_enrollments", "powerschool_sections", "powerschool_enrollments", "powerschool_students"],
    "TeacherAssignment": ["powerschool_sections", "powerschool_teachers", "powerschool_terms", "powerschool_enrollments"],
    "SourceIdentity": ["powerschool_students", "canvas_users", "canvas_enrollments", "moodle_user", "moodle_user_enrolments", "nwea_studentbyschool"],
    "LmsSection": ["canvas_sections", "moodle_course", "powerschool_sections"],
    "CalendarDay": ["powerschool_calendar", "powerschool_schools"],
    "Attendance": ["powerschool_attendance", "powerschool_calendar", "powerschool_enrollments", "powerschool_reenrollments", "powerschool_students", "powerschool_schools"],
    "LearningContent": ["canvas_assignments", "canvas_sections", "moodle_assign", "moodle_course", "powerschool_sections"],
    "Obligation": ["canvas_assignments", "canvas_enrollments", "canvas_sections", "canvas_users", "moodle_assign", "moodle_course", "moodle_enrol", "moodle_user", "moodle_user_enrolments", "powerschool_enrollments", "powerschool_section_enrollments", "powerschool_sections", "powerschool_students"],
    "SubmissionState": ["canvas_assignments", "canvas_enrollments", "canvas_submissions", "canvas_users", "moodle_assign", "moodle_assign_submission", "moodle_enrol", "moodle_user", "moodle_user_enrolments", "powerschool_enrollments", "powerschool_section_enrollments", "powerschool_students"],
    "Submission": ["canvas_assignments", "canvas_enrollments", "canvas_submissions", "canvas_users", "moodle_assign", "moodle_assign_grades", "moodle_assign_submission", "moodle_enrol", "moodle_user", "moodle_user_enrolments", "powerschool_enrollments", "powerschool_section_enrollments", "powerschool_students"],
    "AssessmentInstrument": ["nwea_assessmentresults"],
    "AssessmentWindow": ["nwea_assessmentresults"],
    "AssessmentResult": ["nwea_assessmentresults", "nwea_studentbyschool", "powerschool_students", "powerschool_schools", "powerschool_enrollments", "powerschool_reenrollments"],
    "ActionType": ["powerschool_log_entries"],
    "InterventionAction": ["powerschool_log_entries", "powerschool_students", "powerschool_schools", "powerschool_sections", "powerschool_teachers", "powerschool_districts", "powerschool_courses", "powerschool_calendar", "powerschool_enrollments"],
}
BRONZE_TABLES = sorted({source for sources in TARGET_SOURCES.values() for source in sources})

In [ ]:
def bronze_table(name: str) -> str:
    return f"{bronze_lakehouse_name}.{name}"


def read_bronze(name: str) -> DataFrame:
    table_name = bronze_table(name)
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Missing Bronze table: {table_name}")

    frame = spark.table(table_name)
    if "SourceInstanceId" not in frame.columns:
        raise ValueError(f"Bronze table has no SourceInstanceId: {table_name}")
    selected = frame.filter(F.col("SourceInstanceId") == source_instance_id)
    if selected.limit(1).count() == 0:
        raise ValueError(
            f"Bronze table {table_name} has no rows for SourceInstanceId={source_instance_id}"
        )
    return selected


def delta_version(name: str) -> int:
    table_name = bronze_table(name)
    history = spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").select("version").first()
    if history is None:
        raise ValueError(f"Bronze table has no Delta history: {table_name}")
    return int(history.version)


bronze_versions = {name: delta_version(name) for name in BRONZE_TABLES}
bronze_fingerprint = hashlib.sha256(
    json.dumps(sorted(bronze_versions.items()), separators=(",", ":")).encode()
).hexdigest()

In [ ]:
audit_schema = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("target_table", T.StringType(), False),
    T.StructField("source_instance_id", T.StringType(), False),
    T.StructField("target_fingerprint", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("row_count", T.LongType(), False),
    T.StructField("started_at_utc", T.TimestampType(), False),
    T.StructField("finished_at_utc", T.TimestampType(), False),
    T.StructField("error_message", T.StringType(), True),
])

previous_fingerprints = {}
if spark.catalog.tableExists("transform_audit"):
    latest_window = Window.partitionBy("target_table").orderBy(F.col("finished_at_utc").desc())
    latest_audit = (
        spark.table("transform_audit")
        .filter(
            (F.col("source_instance_id") == source_instance_id)
            & F.col("status").isin("SUCCEEDED", "SKIPPED_UNCHANGED")
        )
        .withColumn("rank", F.row_number().over(latest_window))
        .filter(F.col("rank") == 1)
    )
    previous_fingerprints = {
        row.target_table: row.target_fingerprint
        for row in latest_audit.collect()
    }

target_fingerprints = {}
target_actions = {}
for target, sources in TARGET_SOURCES.items():
    if not spark.catalog.tableExists(target):
        raise ValueError(f"Missing Silver target table: {target}; run EDU_SL_CreateSchema first")

    source_versions = sorted((source, bronze_versions[source]) for source in sources)
    payload = [transform_version, spark.table(target).schema.json(), source_versions]
    fingerprint = hashlib.sha256(json.dumps(payload, separators=(",", ":")).encode()).hexdigest()
    target_fingerprints[target] = fingerprint
    target_actions[target] = (
        "SKIP"
        if not force_reload and previous_fingerprints.get(target) == fingerprint
        else "WRITE"
    )

print(
    f"Targets: {sum(action == 'WRITE' for action in target_actions.values())} to write, "
    f"{sum(action == 'SKIP' for action in target_actions.values())} unchanged"
)
mark_stage("dependencies-complete")

## Deterministic transformation helpers and core entities

In [ ]:
def stable_key(entity: str, *parts) -> Column:
    values = [
        F.lit("education-analytics"),
        F.lit(source_instance_id),
        F.lit(entity),
        *[part.cast("string") for part in parts],
    ]
    return F.sha2(F.to_json(F.array(*values)), 256)


def source_record(*parts) -> Column:
    values = [part.cast("string") for part in parts]
    return F.to_json(F.array(*values)) if len(values) > 1 else values[0]


def unix_timestamp(column: Column) -> Column:
    return F.when(column.cast("long") > 0, F.timestamp_seconds(column.cast("long")))


ps_districts = read_bronze("powerschool_districts")
ps_schools = read_bronze("powerschool_schools")
ps_students = read_bronze("powerschool_students")
ps_teachers = read_bronze("powerschool_teachers")
ps_courses = read_bronze("powerschool_courses")
ps_sections = read_bronze("powerschool_sections")
ps_enrollments = read_bronze("powerschool_enrollments")
ps_section_enrollments = read_bronze("powerschool_section_enrollments")
ps_calendar = read_bronze("powerschool_calendar")
ps_attendance = read_bronze("powerschool_attendance")
ps_terms = read_bronze("powerschool_terms")
ps_reenrollments = read_bronze("powerschool_reenrollments")
canvas_users = read_bronze("canvas_users")
canvas_sections = read_bronze("canvas_sections")
canvas_enrollments = read_bronze("canvas_enrollments")
canvas_assignments = read_bronze("canvas_assignments")
canvas_submissions = read_bronze("canvas_submissions")
moodle_user = read_bronze("moodle_user")
moodle_course = read_bronze("moodle_course")
moodle_enrol = read_bronze("moodle_enrol")
moodle_user_enrolments = read_bronze("moodle_user_enrolments")
moodle_assign = read_bronze("moodle_assign")
moodle_assign_submission = read_bronze("moodle_assign_submission")
moodle_assign_grades = read_bronze("moodle_assign_grades")
nwea_roster = read_bronze("nwea_studentbyschool")
nwea_results = read_bronze("nwea_assessmentresults")
ps_log_entries = read_bronze("powerschool_log_entries")

### Source identifier checks

In [ ]:
def require_identifier_format(
    frame: DataFrame, column: str, pattern: str, source_name: str
) -> None:
    invalid = frame.filter(
        F.col(column).isNull() | ~F.col(column).rlike(pattern)
    ).limit(1).count()
    if invalid:
        raise ValueError(
            f"{source_name}.{column} contains an unsupported identifier format"
        )


require_identifier_format(
    canvas_sections, "sis_section_id", r"^SYN-SECTION-[0-9]+$", "canvas_sections"
)
require_identifier_format(
    moodle_course, "shortname", r"^SYN-SECTION-[0-9]+$", "moodle_course"
)
require_identifier_format(
    nwea_results, "School_StateID", r"^SYN-SCHOOL-[0-9]+$", "nwea_assessmentresults"
)
require_identifier_format(
    ps_log_entries, "entry_date", r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$", "powerschool_log_entries"
)

### Core authority dimensions

In [ ]:
district = ps_districts.select(
    stable_key("District", F.col("district_id")).alias("district_id"),
    F.col("district_name").alias("name"),
    F.col("district_id").alias("source_record_id"),
)
district_map = district.select(
    F.col("source_record_id").alias("source_district_id"),
    "district_id",
)

school = (
    ps_schools
    .join(district_map, ps_schools.district_id == district_map.source_district_id)
    .select(
        stable_key("School", F.col("source_school_id")).alias("school_id"),
        district_map.district_id,
        F.col("school_name").alias("name"),
        F.col("school_code").alias("code"),
        "school_type", "city", "language", "website",
        F.col("source_school_id").alias("source_record_id"),
    )
)
school_map = school.select(
    F.col("source_record_id").alias("source_school_id"),
    "school_id",
)
school_address = (
    ps_schools
    .join(school_map, "source_school_id")
    .select(
        "school_id", "street_address", "city", "state", "postal_code", "county",
        F.col("source_school_id").alias("source_record_id"),
    )
)

student = ps_students.select(
    stable_key("Student", F.col("student_number")).alias("student_id"),
    F.col("first_name").alias("given_name"),
    F.col("last_name").alias("surname"),
    F.col("enrollment_status").alias("status"),
    F.col("student_number").alias("source_record_id"),
)
student_map = student.select(
    F.col("source_record_id").alias("student_number"),
    "student_id",
)

teacher = (
    ps_teachers
    .join(school_map, "source_school_id")
    .select(
        stable_key("Teacher", F.col("source_teacher_id")).alias("teacher_id"),
        "school_id", "name", "department", "role_title",
        F.col("source_teacher_id").alias("source_record_id"),
    )
)
teacher_map = teacher.select(
    F.col("source_record_id").alias("source_teacher_id"),
    "teacher_id",
)

course = ps_courses.select(
    stable_key("Course", F.col("source_course_id")).alias("course_id"),
    F.col("course_name").alias("name"),
    F.col("course_code").alias("code"),
    "description",
    F.col("source_course_id").alias("source_record_id"),
)
course_map = course.select(
    F.col("source_record_id").alias("source_course_id"),
    "course_id",
)

# One term per school year from the SIS terms extract; last_day is inclusive, valid_to is not.
term_bounds = ps_terms.select(
    F.to_date("first_day").alias("valid_from"),
    F.date_add(F.to_date("last_day"), 1).alias("valid_to"),
)
academic_term = term_bounds.select(
    stable_key("AcademicTerm", F.col("valid_from"), F.col("valid_to")).alias("term_id"),
    "valid_from", "valid_to",
)
if academic_term.filter(F.col("valid_from") >= F.col("valid_to")).limit(1).count():
    raise ValueError("powerschool_terms has a term that ends before it starts")
overlapping_terms = (
    academic_term.alias("a").join(
        academic_term.alias("b"),
        (F.col("a.term_id") < F.col("b.term_id"))
        & (F.col("a.valid_from") < F.col("b.valid_to"))
        & (F.col("b.valid_from") < F.col("a.valid_to")),
    ).limit(1).count()
)
if overlapping_terms:
    raise ValueError("powerschool_terms has overlapping school years")

# The SIS extract carries the current year's sections and enrollments; earlier years'
# membership arrives in reenrollments. The current term is the one holding the current
# enrollments, and every section in the extract belongs to it.
current_entry = ps_enrollments.agg(F.max(F.to_date("entry_date")).alias("entry")).first()["entry"]
term_row = academic_term.filter(
    (F.col("valid_from") <= F.lit(current_entry)) & (F.lit(current_entry) < F.col("valid_to"))
).select("term_id", "valid_from", "valid_to")
if term_row.count() != 1:
    raise ValueError("Current enrollments do not fall inside exactly one term in powerschool_terms")

### Sections, memberships, assignments, and calendar

In [ ]:
section = (
    ps_sections
    .join(school_map, "source_school_id")
    .join(course_map, "source_course_id")
    .crossJoin(term_row.select("term_id"))
    .select(
        stable_key("Section", F.col("source_section_id")).alias("section_id"),
        "school_id", "course_id", "term_id",
        F.col("grade_level").alias("grade"),
        F.col("section_name").alias("name"),
        F.col("section_code").alias("code"),
        F.col("source_section_id").alias("source_record_id"),
    )
)
section_map = section.select(
    F.col("source_record_id").alias("source_section_id"),
    "section_id",
    "school_id",
)

# The current year's year group is the student's grade_level; an earlier year's is recorded
# on its reenrollment row.
enrollment_source = (
    ps_enrollments
    .join(ps_students.select("student_number", F.col("grade_level").alias("grade")), "student_number")
    .select("student_number", "source_school_id", "entry_date", "exit_date", "grade",
            F.lit(True).alias("is_current"))
    .unionByName(
        ps_reenrollments.select(
            "student_number", "source_school_id", "entry_date", "exit_date",
            F.col("grade_level").alias("grade"), F.lit(False).alias("is_current"),
        )
    )
)
school_enrollment_with_scope = (
    enrollment_source
    .join(student_map, "student_number")
    .join(school_map, "source_school_id")
    .select(
        stable_key(
            "SchoolEnrollment",
            F.col("student_number"), F.col("source_school_id"), F.col("entry_date"),
        ).alias("enrollment_id"),
        "student_id", "school_id", "grade",
        F.to_date("entry_date").alias("valid_from"),
        F.to_date("exit_date").alias("valid_to"),
        source_record(
            F.col("student_number"), F.col("source_school_id"), F.col("entry_date"),
        ).alias("source_record_id"),
        "is_current",
    )
)
school_enrollment = school_enrollment_with_scope.drop("is_current")
school_enrollment_map = school_enrollment.select(
    "student_id", "school_id", "valid_from", "valid_to",
)
# Class rosters belong to the current year, so they resolve against current membership only.
current_school_enrollment_map = (
    school_enrollment_with_scope.filter("is_current")
    .select("student_id", "school_id", "valid_from", "valid_to")
)

section_enrollment = (
    ps_section_enrollments
    .join(student_map, "student_number")
    .join(section_map, "source_section_id")
    .join(current_school_enrollment_map, ["student_id", "school_id"])
    .select(
        stable_key(
            "SectionEnrollment",
            F.col("student_number"), F.col("source_section_id"), F.col("valid_from"),
        ).alias("enrollment_id"),
        "student_id", "section_id", "valid_from", "valid_to",
        source_record(
            F.col("student_number"), F.col("source_section_id"), F.col("valid_from"),
        ).alias("source_record_id"),
    )
)
section_enrollment_map = section_enrollment.select(
    "student_id", "section_id", "valid_from", "valid_to",
)

teacher_assignment = (
    ps_sections
    .join(teacher_map, "source_teacher_id")
    .join(section_map, "source_section_id")
    .crossJoin(term_row.select("valid_from", "valid_to"))
    .select(
        stable_key(
            "TeacherAssignment",
            F.col("source_section_id"), F.col("source_teacher_id"), F.col("valid_from"),
        ).alias("assignment_id"),
        "teacher_id", "section_id", "valid_from", "valid_to",
        source_record(
            F.col("source_section_id"), F.col("source_teacher_id"), F.col("valid_from"),
        ).alias("source_record_id"),
    )
)

calendar_day = (
    ps_calendar
    .join(school_map, "source_school_id")
    .select(
        stable_key(
            "CalendarDay", F.col("source_school_id"), F.col("instructional_date"),
        ).alias("calendar_id"),
        "school_id",
        F.to_date("instructional_date").alias("instructional_date"),
        F.col("expected_minutes").cast("int").alias("expected_minutes"),
        source_record(F.col("source_school_id"), F.col("instructional_date")).alias("source_record_id"),
    )
)
mark_stage("core-complete")

## Identity, LMS, attendance, work, and assessment transformations

In [ ]:
ps_identity = student_map.select(
    stable_key("SourceIdentity", F.lit("PowerSchool"), F.col("student_number")).alias("mapping_id"),
    "student_id",
    F.lit("PowerSchool").alias("source_name"),
    F.col("student_number").alias("source_user_id"),
    F.col("student_number").alias("source_record_id"),
)

canvas_learners = (
    canvas_enrollments
    .filter((F.col("type") == "StudentEnrollment") & (F.col("enrollment_state") == "active"))
    .select("user_id")
    .distinct()
)
canvas_identity = (
    canvas_users
    .join(canvas_learners, canvas_users.id == canvas_learners.user_id)
    .join(student_map, canvas_users.sis_user_id == student_map.student_number)
    .select(
        stable_key("SourceIdentity", F.lit("Canvas"), canvas_users.id).alias("mapping_id"),
        "student_id",
        F.lit("Canvas").alias("source_name"),
        canvas_users.id.alias("source_user_id"),
        canvas_users.id.alias("source_record_id"),
    )
)

moodle_learners = (
    moodle_user_enrolments.filter(F.col("status") == "0").select("userid").distinct()
)
moodle_identity = (
    moodle_user
    .join(moodle_learners, moodle_user.id == moodle_learners.userid)
    .join(student_map, moodle_user.idnumber == student_map.student_number)
    .select(
        stable_key("SourceIdentity", F.lit("Moodle"), moodle_user.id).alias("mapping_id"),
        "student_id",
        F.lit("Moodle").alias("source_name"),
        moodle_user.id.alias("source_user_id"),
        moodle_user.id.alias("source_record_id"),
    )
)

nwea_identity = (
    nwea_roster.select("StudentID").distinct()
    .join(student_map, F.col("StudentID") == student_map.student_number)
    .select(
        stable_key("SourceIdentity", F.lit("NWEA"), F.col("StudentID")).alias("mapping_id"),
        "student_id",
        F.lit("NWEA").alias("source_name"),
        F.col("StudentID").alias("source_user_id"),
        F.col("StudentID").alias("source_record_id"),
    )
)

source_identity = (
    ps_identity.unionByName(canvas_identity)
    .unionByName(moodle_identity)
    .unionByName(nwea_identity)
)
identity_map = source_identity.select("source_name", "source_user_id", "student_id")

In [ ]:
canvas_section_lookup = canvas_sections.select(
    F.col("id").alias("source_class_id"),
    F.col("course_id").alias("source_course_id"),
)
canvas_lms = (
    canvas_sections
    .withColumn("source_section_id", F.regexp_extract("sis_section_id", r"^SYN-SECTION-([0-9]+)$", 1))
    .join(section_map, "source_section_id")
    .select(
        stable_key("LmsSection", F.lit("Canvas"), canvas_sections.id).alias("mapping_id"),
        "section_id",
        F.lit("Canvas").alias("source_name"),
        canvas_sections.id.alias("source_class_id"),
        canvas_sections.id.alias("source_record_id"),
    )
)
moodle_lms = (
    moodle_course
    .withColumn("source_section_id", F.regexp_extract("shortname", r"^SYN-SECTION-([0-9]+)$", 1))
    .join(section_map, "source_section_id")
    .select(
        stable_key("LmsSection", F.lit("Moodle"), moodle_course.id).alias("mapping_id"),
        "section_id",
        F.lit("Moodle").alias("source_name"),
        moodle_course.id.alias("source_class_id"),
        moodle_course.id.alias("source_record_id"),
    )
)
lms_section = canvas_lms.unionByName(moodle_lms)
lms_map = lms_section.select("source_name", "source_class_id", "section_id")

### Attendance and learning content

In [ ]:
attendance = (
    ps_attendance
    .join(student_map, "student_number")
    .join(school_map, "source_school_id")
    .join(school_enrollment_map, ["student_id", "school_id"])
    .withColumn("attendance_date_typed", F.to_date("attendance_date"))
    .filter(
        (F.col("attendance_date_typed") >= F.col("valid_from"))
        & (F.col("attendance_date_typed") < F.col("valid_to"))
    )
    .select(
        stable_key(
            "Attendance", F.col("student_number"),
            F.col("source_school_id"), F.col("attendance_date"),
        ).alias("attendance_id"),
        "student_id", "school_id",
        F.col("attendance_date_typed").alias("attendance_date"),
        F.when(F.col("attendance_code") == "P", "present")
        .when(F.col("attendance_code") == "A", "absent").alias("status"),
        F.col("minutes_absent").cast("int"),
        source_record(
            F.col("student_number"), F.col("source_school_id"),
            F.col("attendance_date"),
        ).alias("source_record_id"),
    )
)

canvas_assignment = canvas_assignments.alias("assignment")
canvas_class = canvas_section_lookup.alias("class")
canvas_lms_map = lms_map.filter(F.col("source_name") == "Canvas").alias("lms")
canvas_content = (
    canvas_assignment
    .join(canvas_class, F.col("assignment.course_id") == F.col("class.source_course_id"))
    .join(canvas_lms_map, F.col("class.source_class_id") == F.col("lms.source_class_id"))
    .select(
        stable_key("LearningContent", F.lit("Canvas"), F.col("assignment.id")).alias("content_id"),
        F.col("lms.section_id").alias("section_id"),
        F.col("assignment.name").alias("name"),
        F.col("assignment.description").alias("description"),
        F.to_timestamp(F.col("assignment.due_at")).alias("due_at"),
        F.when(
            (F.col("assignment.grading_type") == "points")
            & (F.col("assignment.points_possible").cast("double") > 0),
            F.col("assignment.points_possible").cast("double"),
        ).alias("max_score"),
        F.lit("Canvas").alias("source_name"),
        F.col("assignment.id").alias("source_record_id"),
    )
)

moodle_assignment = moodle_assign.alias("assignment")
moodle_class = moodle_course.select(F.col("id").alias("source_class_id")).alias("class")
moodle_lms_map = lms_map.filter(F.col("source_name") == "Moodle").alias("lms")
moodle_content = (
    moodle_assignment
    .join(moodle_class, F.col("assignment.course") == F.col("class.source_class_id"))
    .join(moodle_lms_map, F.col("class.source_class_id") == F.col("lms.source_class_id"))
    .select(
        stable_key("LearningContent", F.lit("Moodle"), F.col("assignment.id")).alias("content_id"),
        F.col("lms.section_id").alias("section_id"),
        F.col("assignment.name").alias("name"),
        F.col("assignment.intro").alias("description"),
        unix_timestamp(F.col("assignment.duedate")).alias("due_at"),
        F.when(F.col("assignment.grade").cast("double") > 0, F.col("assignment.grade").cast("double")).alias("max_score"),
        F.lit("Moodle").alias("source_name"),
        F.col("assignment.id").alias("source_record_id"),
    )
)
learning_content = canvas_content.unionByName(moodle_content)
content_map = learning_content.select(
    "source_name",
    F.col("source_record_id").alias("source_content_id"),
    "content_id", "section_id",
    F.col("due_at").alias("effective_due_at"),
)

### Student obligations

In [ ]:
canvas_students = (
    canvas_enrollments.alias("enrollment")
    .filter(
        (F.col("enrollment.type") == "StudentEnrollment")
        & (F.col("enrollment.enrollment_state") == "active")
    )
    .join(
        identity_map.filter(F.col("source_name") == "Canvas").alias("identity"),
        F.col("enrollment.user_id") == F.col("identity.source_user_id"),
    )
    .select(
        F.col("enrollment.course_id").alias("source_course_id"),
        F.col("identity.student_id").alias("student_id"),
    )
)
canvas_assignment_lookup = canvas_assignments.select(
    F.col("id").alias("source_content_id"),
    F.col("course_id").alias("source_course_id"),
)
canvas_content_map = (
    content_map.filter(F.col("source_name") == "Canvas")
    .select("source_content_id", "content_id", "section_id", "effective_due_at")
)
canvas_obligation = (
    canvas_students
    .join(canvas_assignment_lookup, "source_course_id")
    .join(canvas_content_map, "source_content_id")
    .join(section_enrollment_map, ["student_id", "section_id"])
    .select(
        stable_key(
            "Obligation", F.col("student_id"), F.col("content_id"), F.col("valid_from"),
        ).alias("obligation_id"),
        "student_id", "content_id", "section_id",
        F.col("effective_due_at").alias("due_at"),
        "valid_from", "valid_to",
    )
)

moodle_students = (
    moodle_user_enrolments.alias("user_enrollment")
    .filter(F.col("user_enrollment.status") == "0")
    .join(
        identity_map.filter(F.col("source_name") == "Moodle").alias("identity"),
        F.col("user_enrollment.userid") == F.col("identity.source_user_id"),
    )
    .join(
        moodle_enrol.select(
            F.col("id").alias("enrolid"),
            F.col("courseid").alias("source_course_id"),
        ),
        "enrolid",
    )
    .select("student_id", "source_course_id")
)
moodle_assignment_lookup = moodle_assign.select(
    F.col("id").alias("source_content_id"),
    F.col("course").alias("source_course_id"),
)
moodle_content_map = (
    content_map.filter(F.col("source_name") == "Moodle")
    .select("source_content_id", "content_id", "section_id", "effective_due_at")
)
moodle_obligation = (
    moodle_students
    .join(moodle_assignment_lookup, "source_course_id")
    .join(moodle_content_map, "source_content_id")
    .join(section_enrollment_map, ["student_id", "section_id"])
    .select(
        stable_key(
            "Obligation", F.col("student_id"), F.col("content_id"), F.col("valid_from"),
        ).alias("obligation_id"),
        "student_id", "content_id", "section_id",
        F.col("effective_due_at").alias("due_at"),
        "valid_from", "valid_to",
    )
)

obligation = canvas_obligation.unionByName(moodle_obligation)
obligation_map = obligation.select("obligation_id", "student_id", "content_id")

### Submission state and attempts

In [ ]:
canvas_submission_source = canvas_submissions.alias("submission")
canvas_submission_identity = (
    identity_map.filter(F.col("source_name") == "Canvas")
    .select("source_user_id", "student_id")
    .alias("identity")
)
canvas_submission_content = (
    content_map.filter(F.col("source_name") == "Canvas")
    .select("source_content_id", "content_id")
    .alias("content")
)
canvas_work = (
    canvas_submission_source
    .join(
        canvas_submission_identity,
        F.col("submission.user_id") == F.col("identity.source_user_id"),
    )
    .join(
        canvas_submission_content,
        F.col("submission.assignment_id") == F.col("content.source_content_id"),
    )
    .select(
        F.col("submission.assignment_id").alias("source_assignment_id"),
        F.col("submission.user_id").alias("source_user_id"),
        F.col("submission.attempt").alias("attempt"),
        F.col("submission.submitted_at").alias("submitted_at"),
        F.col("submission.score").alias("score"),
        F.col("submission.workflow_state").alias("workflow_state"),
        F.col("submission.late").alias("late"),
        F.col("submission.missing").alias("missing"),
        "student_id", "content_id",
    )
    .join(obligation_map, ["student_id", "content_id"])
)

moodle_submission_source = moodle_assign_submission.alias("submission")
moodle_submission_identity = (
    identity_map.filter(F.col("source_name") == "Moodle")
    .select("source_user_id", "student_id")
    .alias("identity")
)
moodle_submission_content = (
    content_map.filter(F.col("source_name") == "Moodle")
    .select("source_content_id", "content_id")
    .alias("content")
)
moodle_work = (
    moodle_submission_source
    .join(
        moodle_submission_identity,
        F.col("submission.userid") == F.col("identity.source_user_id"),
    )
    .join(
        moodle_submission_content,
        F.col("submission.assignment") == F.col("content.source_content_id"),
    )
    .select(
        F.col("submission.id").alias("source_submission_id"),
        F.col("submission.assignment").alias("source_assignment_id"),
        F.col("submission.userid").alias("source_user_id"),
        F.col("submission.attemptnumber").alias("attempt_number"),
        F.col("submission.status").alias("submission_status"),
        F.col("submission.timemodified").alias("submission_modified_at"),
        "student_id", "content_id",
    )
    .join(obligation_map, ["student_id", "content_id"])
)

canvas_state = canvas_work.select(
    stable_key(
        "SubmissionState", F.lit("Canvas"),
        F.col("source_assignment_id"), F.col("source_user_id"),
    ).alias("state_id"),
    "obligation_id",
    F.col("workflow_state").alias("state"),
    F.col("missing").cast("boolean").alias("source_missing"),
    F.col("late").cast("boolean").alias("source_late"),
    source_record(F.col("source_assignment_id"), F.col("source_user_id")).alias("source_record_id"),
    F.lit(None).cast("timestamp").alias("source_modified_at"),
)
moodle_state = moodle_work.select(
    stable_key("SubmissionState", F.lit("Moodle"), F.col("source_submission_id")).alias("state_id"),
    "obligation_id",
    F.col("submission_status").alias("state"),
    F.lit(None).cast("boolean").alias("source_missing"),
    F.lit(None).cast("boolean").alias("source_late"),
    F.col("source_submission_id").alias("source_record_id"),
    unix_timestamp(F.col("submission_modified_at")).alias("source_modified_at"),
)
submission_state = canvas_state.unionByName(moodle_state)

In [ ]:
canvas_submission = (
    canvas_work
    .filter(F.col("attempt").isNotNull() & F.col("submitted_at").isNotNull())
    .select(
        stable_key(
            "Submission", F.lit("Canvas"), F.col("source_assignment_id"),
            F.col("source_user_id"), F.col("attempt"),
        ).alias("submission_id"),
        "obligation_id", "student_id", "content_id",
        F.col("attempt").cast("int").alias("attempt"),
        F.to_timestamp("submitted_at").alias("submitted_at"),
        F.col("score").cast("double").alias("score"),
        F.col("workflow_state").alias("state"),
        F.col("late").cast("boolean").alias("source_late"),
        F.col("missing").cast("boolean").alias("source_missing"),
        source_record(
            F.col("source_assignment_id"), F.col("source_user_id"), F.col("attempt"),
        ).alias("source_record_id"),
        F.lit(None).cast("timestamp").alias("source_modified_at"),
    )
)

moodle_grades = moodle_assign_grades.select(
    F.col("assignment").alias("grade_assignment_id"),
    F.col("userid").alias("grade_user_id"),
    F.col("attemptnumber").alias("grade_attempt"),
    F.col("grade").cast("double").alias("score"),
)
moodle_submission = (
    moodle_work
    .filter(F.col("submission_status") == "submitted")
    .join(
        moodle_grades,
        (F.col("source_assignment_id") == F.col("grade_assignment_id"))
        & (F.col("source_user_id") == F.col("grade_user_id"))
        & (F.col("attempt_number") == F.col("grade_attempt")),
        "left",
    )
    .select(
        stable_key(
            "Submission", F.lit("Moodle"),
            F.col("source_submission_id"), F.col("attempt_number"),
        ).alias("submission_id"),
        "obligation_id", "student_id", "content_id",
        F.col("attempt_number").cast("int").alias("attempt"),
        unix_timestamp(F.col("submission_modified_at")).alias("submitted_at"),
        "score",
        F.col("submission_status").alias("state"),
        F.lit(None).cast("boolean").alias("source_late"),
        F.lit(None).cast("boolean").alias("source_missing"),
        F.col("source_submission_id").alias("source_record_id"),
        unix_timestamp(F.col("submission_modified_at")).alias("source_modified_at"),
    )
)
submission = canvas_submission.unionByName(moodle_submission)

### Assessment instruments, windows, and results

In [ ]:
assessment_instrument = (
    nwea_results.select("TestName", "Subject").distinct()
    .select(
        stable_key("AssessmentInstrument", F.col("TestName"), F.col("Subject")).alias("assessment_id"),
        F.col("TestName").alias("name"),
        F.col("Subject").alias("subject"),
        F.lit("RIT").alias("unit"),
        source_record(F.col("TestName"), F.col("Subject")).alias("source_record_id"),
    )
)
instrument_map = assessment_instrument.select(
    F.col("name").alias("TestName"),
    F.col("subject").alias("Subject"),
    "assessment_id",
)

window_bounds = nwea_results.groupBy("TermName").agg(
    F.min(F.to_date("TestStartDate")).alias("start_date"),
    F.max(F.to_date("TestStartDate")).alias("end_date"),
)
if (
    window_bounds.groupBy("start_date").count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise ValueError("Assessment windows have ambiguous start-date ordering")

assessment_window = (
    window_bounds
    .withColumn("sequence", F.dense_rank().over(Window.orderBy("start_date")))
    .select(
        stable_key("AssessmentWindow", F.col("TermName")).alias("window_id"),
        F.col("TermName").alias("name"),
        "start_date", "end_date",
        F.col("sequence").cast("int").alias("sequence"),
        F.col("TermName").alias("source_record_id"),
    )
)
window_map = assessment_window.select(F.col("name").alias("TermName"), "window_id")

assessment_result = (
    nwea_results
    .join(
        identity_map.filter(F.col("source_name") == "NWEA"),
        F.col("StudentID") == F.col("source_user_id"),
    )
    .withColumn(
        "source_school_id",
        F.regexp_extract("School_StateID", r"^SYN-SCHOOL-([0-9]+)$", 1),
    )
    .join(school_map, "source_school_id")
    .join(instrument_map, ["TestName", "Subject"])
    .join(window_map, "TermName")
    .join(school_enrollment_map, ["student_id", "school_id"])
    .withColumn("assessment_date_typed", F.to_date("TestStartDate"))
    .filter(
        (F.col("assessment_date_typed") >= F.col("valid_from"))
        & (F.col("assessment_date_typed") < F.col("valid_to"))
    )
    .select(
        stable_key("AssessmentResult", F.col("TestID")).alias("result_id"),
        "student_id", "school_id", "assessment_id", "window_id",
        F.col("assessment_date_typed").alias("assessment_date"),
        F.col("TestRITScore").cast("double").alias("score"),
        F.col("TestStandardError").cast("double").alias("standard_error"),
        F.col("TestID").alias("source_record_id"),
    )
)

### Teacher-logged actions

One row per action a teacher logged for a learner, such as a check-in, a family contact or a
missing-work follow-up. The action type is a governed list read from the source; the reason
category is the part of the recorded reason before its first colon (`Attendance`,
`Coursework`), so a district can add rules without a code change. Status is normalized to
`open` or `completed`. A class is optional: a school-wide action such as a counselor
referral may not belong to one. Nothing here scores or predicts; the outcome note is the
teacher's own descriptive record.

In [ ]:
action_type = (
    ps_log_entries.select("log_type").distinct()
    .select(
        stable_key("ActionType", F.col("log_type")).alias("action_type_id"),
        F.col("log_type").alias("name"),
        F.col("log_type").alias("source_record_id"),
    )
)
action_type_map = action_type.select(F.col("name").alias("log_type"), "action_type_id")

unmapped_status = (
    ps_log_entries.filter(~F.lower(F.col("status")).isin("open", "completed"))
    .limit(1).count()
)
if unmapped_status:
    raise ValueError("powerschool_log_entries.status contains a value other than Open or Completed")

reason_category = F.trim(F.split(F.col("reason"), ":").getItem(0))
intervention_action = (
    ps_log_entries
    .join(student_map, "student_number")
    .join(school_map, "source_school_id")
    .join(teacher_map, "source_teacher_id")
    .join(action_type_map, "log_type")
    .join(
        section_map.select("source_section_id", F.col("section_id")),
        "source_section_id",
        "left",
    )
    .select(
        stable_key("InterventionAction", F.col("log_id")).alias("action_id"),
        "student_id", "school_id", "section_id", "teacher_id", "action_type_id",
        F.to_date("entry_date").alias("action_date"),
        F.col("reason"),
        F.when(F.col("reason").contains(":"), reason_category).otherwise(F.lit("Other")).alias("reason_category"),
        F.lower(F.col("status")).alias("status"),
        F.to_date("follow_up_date").alias("follow_up_date"),
        F.to_date("completed_date").alias("completed_date"),
        F.col("outcome_note"),
        F.col("log_id").alias("source_record_id"),
    )
)

In [ ]:
TARGETS = {
    "District": district,
    "School": school,
    "SchoolAddress": school_address,
    "Student": student,
    "Teacher": teacher,
    "Course": course,
    "AcademicTerm": academic_term,
    "Section": section,
    "SchoolEnrollment": school_enrollment,
    "SectionEnrollment": section_enrollment,
    "TeacherAssignment": teacher_assignment,
    "SourceIdentity": source_identity,
    "LmsSection": lms_section,
    "CalendarDay": calendar_day,
    "Attendance": attendance,
    "LearningContent": learning_content,
    "Obligation": obligation,
    "SubmissionState": submission_state,
    "Submission": submission,
    "AssessmentInstrument": assessment_instrument,
    "AssessmentWindow": assessment_window,
    "AssessmentResult": assessment_result,
    "ActionType": action_type,
    "InterventionAction": intervention_action,
}
mark_stage("domain-complete")

### Source-to-target completeness

In [ ]:
EXPECTED_ROW_COUNTS = {
    "District": lambda: ps_districts.count(),
    "School": lambda: ps_schools.count(),
    "Student": lambda: ps_students.count(),
    "Teacher": lambda: ps_teachers.count(),
    "Course": lambda: ps_courses.count(),
    "SchoolAddress": lambda: ps_schools.count(),
    "AcademicTerm": lambda: ps_terms.count(),
    "Section": lambda: ps_sections.count(),
    "SchoolEnrollment": lambda: ps_enrollments.count() + ps_reenrollments.count(),
    "SectionEnrollment": lambda: ps_section_enrollments.count(),
    "TeacherAssignment": lambda: ps_sections.count(),
    "SourceIdentity": lambda: (
        ps_students.count()
        + canvas_learners.count()
        + moodle_learners.count()
        + nwea_roster.select("StudentID").distinct().count()
    ),
    "LmsSection": lambda: canvas_sections.count() + moodle_course.count(),
    "CalendarDay": lambda: ps_calendar.count(),
    "Attendance": lambda: ps_attendance.count(),
    "LearningContent": lambda: canvas_assignments.count() + moodle_assign.count(),
    "Obligation": lambda: (
        canvas_students.join(canvas_assignment_lookup, "source_course_id").count()
        + moodle_students.join(moodle_assignment_lookup, "source_course_id").count()
    ),
    "SubmissionState": lambda: (
        canvas_submissions.count() + moodle_assign_submission.count()
    ),
    "Submission": lambda: (
        canvas_submissions.filter(
            F.col("attempt").isNotNull() & F.col("submitted_at").isNotNull()
        ).count()
        + moodle_assign_submission.filter(F.col("status") == "submitted").count()
    ),
    "AssessmentInstrument": lambda: (
        nwea_results.select("TestName", "Subject").distinct().count()
    ),
    "AssessmentWindow": lambda: nwea_results.select("TermName").distinct().count(),
    "AssessmentResult": lambda: nwea_results.count(),
    "ActionType": lambda: ps_log_entries.select("log_type").distinct().count(),
    "InterventionAction": lambda: ps_log_entries.count(),
}

if set(EXPECTED_ROW_COUNTS) != set(TARGETS):
    raise ValueError("Source completeness checks do not cover every Silver target")

## Conform, validate, and publish

In [ ]:
def conform(frame: DataFrame, table_name: str) -> DataFrame:
    schema = spark.table(table_name).schema
    missing = [field.name for field in schema if field.name not in frame.columns]
    extra = [name for name in frame.columns if name not in schema.fieldNames()]
    if missing or extra:
        raise ValueError(f"{table_name} columns differ; missing={missing}, extra={extra}")

    return frame.select(*[
        F.col(field.name).cast(field.dataType).alias(field.name)
        for field in schema
    ])


changed = {
    name: conform(frame, name).cache()
    for name, frame in TARGETS.items()
    if target_actions[name] == "WRITE"
}

### Required values and primary keys

In [ ]:
quality_errors = []
row_counts = {}

for name, frame in changed.items():
    schema = spark.table(name).schema
    row_counts[name] = frame.count()
    expected_count = EXPECTED_ROW_COUNTS[name]()
    if row_counts[name] != expected_count:
        quality_errors.append(
            f"{name} source rows expected {expected_count}, received {row_counts[name]}"
        )

    primary_keys = [
        field.name
        for field in schema
        if field.metadata.get("primaryKey")
    ]
    required = [
        field.name
        for field in schema
        if not field.metadata.get("declaredNullable", True)
    ]
    null_counts = frame.agg(*[
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in required
    ]).first().asDict()
    bad_nulls = {
        column: value
        for column, value in null_counts.items()
        if value
    }
    if bad_nulls:
        quality_errors.append(f"{name} required nulls: {bad_nulls}")

    for key in primary_keys:
        duplicates = (
            frame.groupBy(key).count()
            .filter(F.col("count") > 1)
            .limit(1)
            .count()
        )
        if duplicates:
            quality_errors.append(f"{name} duplicate primary key: {key}")

### Foreign keys and quality gate

In [ ]:
for child_name, child_frame in changed.items():
    for field in spark.table(child_name).schema:
        foreign_key = field.metadata.get("foreignKey")
        if not foreign_key:
            continue

        parent_name, parent_column = foreign_key.split(".", 1)
        parent_frame = changed[parent_name] if parent_name in changed else spark.table(parent_name)
        parent_keys = parent_frame.select(
            F.col(parent_column).alias(field.name)
        ).distinct()
        unresolved = (
            child_frame.select(field.name).distinct()
            .filter(F.col(field.name).isNotNull())
            .join(parent_keys, field.name, "left_anti")
            .limit(1)
            .count()
        )
        if unresolved:
            quality_errors.append(
                f"{child_name}.{field.name} has unresolved {foreign_key}"
            )

if quality_errors:
    for frame in changed.values():
        frame.unpersist()

    raise ValueError(
        "Silver quality validation failed:\n- " + "\n- ".join(quality_errors)
    )

### Publication and audit

In [ ]:
audit_rows = []
for name in TARGETS:
    started_at = datetime.now(timezone.utc).replace(tzinfo=None)
    if target_actions[name] == "SKIP":
        count = spark.table(name).count()
        status = "SKIPPED_UNCHANGED"
    else:
        changed[name].write.mode("overwrite").insertInto(name, overwrite=True)
        count = row_counts[name]
        status = "SUCCEEDED"
        changed[name].unpersist()

    audit_rows.append((
        run_id,
        name,
        source_instance_id,
        target_fingerprints[name],
        status,
        count,
        started_at,
        datetime.now(timezone.utc).replace(tzinfo=None),
        None,
    ))

audit_frame = spark.createDataFrame(audit_rows, audit_schema)
audit_frame.write.format("delta").mode("append").saveAsTable("transform_audit")
display(audit_frame.orderBy("target_table"))

In [ ]:
result = {
    "status": "SUCCEEDED",
    "runId": run_id,
    "sourceInstanceId": source_instance_id,
    "bronzeFingerprint": bronze_fingerprint,
    "tablesExpected": len(TARGETS),
    "tablesWritten": sum(value == "WRITE" for value in target_actions.values()),
    "tablesSkipped": sum(value == "SKIP" for value in target_actions.values()),
    "tablesFailed": 0,
    "rowsPublished": sum(row_counts.values()),
    "qualityErrors": 0,
}
result_json = json.dumps(result, separators=(",", ":"))
display(spark.createDataFrame([result]))
print(result_json)

if exit_notebook_on_success:
    notebookutils.notebook.exit(result_json)